# Stage 1: train the GDN/residual transform on Vimeo-90K

`PARITY_ROADMAP.md` Stage 1. The Stage 0 scoreboard located the gap to H.264 in the
transform rather than the entropy coder, and `nvc.models.ResidualGDNAutoencoder`
(8,437,827 parameters, GDN/IGDN + residual blocks) is the replacement. This notebook
trains it.

**This notebook is a thin wrapper.** All the logic lives in
`scripts/train_vimeo_stage1.py`, which is tested in CI. The notebook mounts Drive,
supplies Kaggle credentials, installs the package, and runs that script. Anything you
want to change lives in the config cell below or in the script's own flags
(`!python scripts/train_vimeo_stage1.py --help`).

## The two phases, and why there are two

The rate proxy needs a quantization bin width, and a bin width comes from calibrating
a *trained* model — a from-scratch network has no meaningful latent scale to calibrate.
So this runs the way the baseline lineage did (M7/M8 distortion-only, then M9+ rate):

1. **Phase A** — distortion only, from scratch, over all ten chunks.
2. **Calibrate** the Phase A result on a train-split manifest.
3. **Phase B** — `D + lambda*R`, continuing from Phase A's weights.

Run Phase A to completion before touching Phase B. Sections are ordered so you can
just run top to bottom over several sessions.

## Before you start

- A Kaggle account and API token (kaggle.com/settings → API → Create New Token). You
  upload it once; it is cached on Drive so reconnects do not ask again.
- A Google account for Drive — checkpoints and progress live there, so a disconnect
  costs at most one chunk's epoch.
- A GPU runtime (Runtime → Change runtime type → GPU). Check the sanity cell's
  reported step time before committing to a long run.

## If the session disconnects

Reopen and Run All. `progress.json` on Drive records which chunks are done and they
are skipped; training resumes from `checkpoints/latest.pt` with its optimizer state.
Nothing is lost beyond the epoch in flight.

## What this does *not* do

Measure anything. The Stage 1 gate is intra-only BD-rate from
`scripts/benchmark_intra_gate.py`, against the denominator already measured in
`outputs/benchmarks/parity_intra/` (**+176.2%** PSNR for the current codec). A
checkpoint from here is an input to that measurement, not a result. Section 7 has
the exact command.

## 0. Configuration

In [ ]:
from pathlib import Path

# --- Which Kaggle chunks, in order (wangsally/vimeo-90k-1 .. -10) ---
CHUNKS = list(range(1, 11))        # the full ~89GB dataset; use e.g. [1, 2] for a short trial

# --- Where things live ---
REPO_URL = 'https://github.com/yuvidewan/neural_streaming.git'
REPO_DIR = Path('/content/neural_streaming')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/neural_streaming_colab')
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'stage1_vimeo'   # checkpoints/, progress.json, history.json
SCRATCH_DIR = Path('/content/vimeo_scratch')      # one chunk at a time, wiped as it goes

# --- Training ---
BATCH_SIZE = 16           # the Stage 1 model is 14x the baseline; 16 keeps a T4 comfortable
CROP_SIZE = 256           # Vimeo frames are 448x256; must be divisible by 16
LEARNING_RATE = 1e-4
EPOCHS_PER_CHUNK_MAX = 4  # ceiling; early stopping usually stops sooner
SEED = 42
NUM_WORKERS = 2           # Colab is Linux, so >0 is safe

# --- Architecture. Keep these IDENTICAL across every run you resume from:
# changing one makes the saved checkpoint unloadable. ---
LATENT_CHANNELS = 192
BASE_CHANNELS = 192
RESIDUAL_BLOCKS = 1

# --- Phase B (the rate objective). Ignored until you reach Section 5. ---
CALIBRATION_BITS = 4
CALIBRATION_MODE = 'per_channel'
CALIBRATION_CHUNK = 1     # re-downloaded just to calibrate on; chunks are deleted after training
CALIBRATION_PATH = OUTPUT_DIR / 'calibration' / f'stage1_{CALIBRATION_BITS}bit_train.json'
RATE_LAMBDA = 0.01        # the rate weight in D + lambda*R; sweep this, it is not a settled value
RATE_TRACK_SCALE = True   # M9F.5: stops the encoder gaming a frozen bin width by shrinking the latent

KAGGLE_DATASET_OWNER = 'wangsally'
KAGGLE_DATASET_PREFIX = 'vimeo-90k'

print(f'Output (on Drive): {OUTPUT_DIR}')
print(f'Chunks: {CHUNKS}')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'calibration').mkdir(parents=True, exist_ok=True)
print(f'Checkpoints and progress will live in {OUTPUT_DIR}')
print('Note this is a DIFFERENT directory from the baseline runs\' checkpoints/ -')
print('a shared progress.json would make this run skip chunks it never trained on.')

## 2. Kaggle API credentials

Uploaded once, then cached on Drive. Reconnects reuse the cached copy.

In [ ]:
import os
import shutil
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_target = kaggle_dir / 'kaggle.json'
drive_kaggle_json = DRIVE_PROJECT_DIR / 'kaggle.json'

if drive_kaggle_json.is_file():
    shutil.copy(drive_kaggle_json, kaggle_json_target)
    print(f'Reused Kaggle API token from {drive_kaggle_json}')
else:
    print('No saved Kaggle token found on Drive - upload your kaggle.json now.')
    print('(Get it from https://www.kaggle.com/settings -> API -> Create New Token)')
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    shutil.move(uploaded_name, kaggle_json_target)
    shutil.copy(kaggle_json_target, drive_kaggle_json)
    print(f'Saved a copy to {drive_kaggle_json} - future reconnects reuse it.')

os.chmod(kaggle_json_target, 0o600)

## 3. Get the project code and install it

In [ ]:
import os
import subprocess

if REPO_DIR.is_dir():
    print(f'{REPO_DIR} already exists - pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q kaggle

### Sanity check

Fails loudly here rather than several cells in. Also reports the measured step time,
which is what tells you whether this runtime is worth a long run: on an RTX 5060 the
Stage 1 model runs ~191 ms/step at batch 8, against the baseline's 8 ms.

In [ ]:
import time
import torch
import nvc
from nvc.models import ResidualGDNAutoencoder

print(f'nvc loaded from: {nvc.__file__}')
print(f'CUDA available: {torch.cuda.is_available()}',
      f'| {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else '(CPU ONLY - stop and switch to a GPU runtime)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResidualGDNAutoencoder(latent_channels=LATENT_CHANNELS, base_channels=BASE_CHANNELS,
                               residual_blocks=RESIDUAL_BLOCKS).to(device).train()
print(f'{type(model).__name__}: {model.num_parameters():,} parameters')

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
batch = torch.rand(BATCH_SIZE, 3, CROP_SIZE, CROP_SIZE, device=device)
for _ in range(3):  # warm up: first steps pay for kernel selection and allocation
    optimizer.zero_grad(); torch.nn.functional.mse_loss(model(batch), batch).backward(); optimizer.step()
if device.type == 'cuda':
    torch.cuda.synchronize()
started = time.time()
for _ in range(10):
    optimizer.zero_grad(); torch.nn.functional.mse_loss(model(batch), batch).backward(); optimizer.step()
if device.type == 'cuda':
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / 2**30
    print(f'Peak GPU memory: {peak:.2f} GiB')
print(f'{(time.time() - started) / 10 * 1000:.0f} ms/step at batch {BATCH_SIZE} x {CROP_SIZE}x{CROP_SIZE}')

del model, optimizer, batch
if device.type == 'cuda':
    torch.cuda.empty_cache()

## 4. Phase A — distortion-only training

From scratch over every chunk in `CHUNKS`. Re-running this cell after a disconnect
resumes; finished chunks are skipped.

Expect this to be the long part. One chunk is a 6–10GB download plus up to
`EPOCHS_PER_CHUNK_MAX` epochs over it, and there are ten of them.

In [ ]:
import subprocess
import sys

COMMON = [
    sys.executable, 'scripts/train_vimeo_stage1.py',
    '--output-dir', str(OUTPUT_DIR),
    '--scratch-dir', str(SCRATCH_DIR),
    '--chunks', *[str(c) for c in CHUNKS],
    '--batch-size', str(BATCH_SIZE),
    '--crop-size', str(CROP_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--epochs-per-chunk-max', str(EPOCHS_PER_CHUNK_MAX),
    '--latent-channels', str(LATENT_CHANNELS),
    '--base-channels', str(BASE_CHANNELS),
    '--residual-blocks', str(RESIDUAL_BLOCKS),
    '--num-workers', str(NUM_WORKERS),
    '--seed', str(SEED),
    '--kaggle-dataset-owner', KAGGLE_DATASET_OWNER,
    '--kaggle-dataset-prefix', KAGGLE_DATASET_PREFIX,
]

# An argument list rather than a shell string: no quoting to get wrong when a
# Drive path contains a space, and check=True means a failure stops the cell
# instead of scrolling past.
subprocess.run(COMMON, check=True)

### Where Phase A got to

In [ ]:
import json

progress = json.loads((OUTPUT_DIR / 'progress.json').read_text())
history = json.loads((OUTPUT_DIR / 'checkpoints' / 'history.json').read_text())
print(f"Completed chunks: {progress['completed_chunks']}")
print(f"Best validation loss: {progress['best_val_loss']}")
print(f"Epochs recorded: {len(history)}")
for record in history[-5:]:
    print(f"  epoch {record['epoch']:3d} chunk {record['chunk']:2d}  "
          f"train {record['train_loss']:.6f}  val {record['val_loss']:.6f}  "
          f"psnr {record['val_psnr']:.2f} dB")

## 5. Calibrate the trained model

Only after Phase A has finished every chunk. The rate proxy's bin width comes from
this file, and `QuantizationNoise.from_calibration` refuses anything not computed on
the train split.

Chunks are deleted as Phase A goes, so their frames are gone and the old manifests
point at nothing. This cell re-downloads one chunk purely to calibrate on, using the
same chunk machinery the trainer uses.

In [ ]:
import importlib.util
import subprocess
import sys

spec = importlib.util.spec_from_file_location(
    'train_vimeo_stage1', REPO_DIR / 'scripts' / 'train_vimeo_stage1.py')
stage1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(stage1)
chunks = stage1._load_script('train_vimeo_qat_combined')

vimeo_root = SCRATCH_DIR / 'vimeo_root'
group_root = chunks.download_and_extract_chunk(
    CALIBRATION_CHUNK, SCRATCH_DIR / f'calib_chunk_{CALIBRATION_CHUNK}',
    dataset_owner=KAGGLE_DATASET_OWNER, dataset_prefix=KAGGLE_DATASET_PREFIX)
chunks.relink_sequences_to_chunk(group_root, vimeo_root)
sequence_ids = chunks.discover_complete_sequence_ids(vimeo_root / 'sequences')
chunks.write_chunk_split_lists(vimeo_root, sequence_ids, SEED)
train_manifest, _ = chunks.build_chunk_manifests(
    vimeo_root, OUTPUT_DIR / 'calib_manifest.json', SEED)

subprocess.run([
    sys.executable, 'scripts/calibrate_quantizer.py',
    '--checkpoint', str(OUTPUT_DIR / 'checkpoints' / 'best.pt'),
    '--manifest', str(train_manifest),
    '--bits', str(CALIBRATION_BITS),
    '--mode', CALIBRATION_MODE,
    '--output', str(CALIBRATION_PATH),
], check=True)
print(f'\nCalibration written to {CALIBRATION_PATH}')

## 6. Phase B — the rate objective

`D + lambda*R`, continuing from Phase A's weights. `--reset-progress` clears the
completed-chunk list so every chunk is visited again; the weights are kept.

The optimizer's parameter list changes between phases (the rate estimator's own
`loc`/`log_scale` join it), so the script starts the optimizer fresh here and says so
— that is expected, not an error.

**`RATE_LAMBDA` is not a settled value.** It trades rate against distortion and wants
a sweep, not a single guess. Start by running a couple of chunks at two or three
values and comparing, rather than committing ten chunks to one number.

In [ ]:
import subprocess

phase_b = COMMON + [
    '--rate-enabled',
    '--rate-calibration', str(CALIBRATION_PATH),
    '--rate-lambda', str(RATE_LAMBDA),
    '--reset-progress',
]
if RATE_TRACK_SCALE:
    phase_b.append('--rate-track-scale')

subprocess.run(phase_b, check=True)

## 7. Take the checkpoint home

`best.pt` is already on Drive and safe. This downloads it to your machine so the gate
can be measured locally (the gate needs the DAVIS frames, which are not on Colab).

### Then, on your machine

The checkpoint is only an input. Measure the gate with `benchmark_intra_gate.py`:

```bash
python scripts/benchmark_intra_gate.py     --checkpoint <the downloaded best.pt>     --output-dir outputs/benchmarks/intra_gate_stage1     --output-name intra_gate_stage1.json
```

Compare its headline BD-rate against the current codec's **+176.2%** in
`outputs/benchmarks/parity_intra/README.md`. Lower is better; a *negative* number
would mean the Stage 1 transform beats x264 all-intra, which is not the near-term
expectation - the target for this stage is a large cut in that +176.2%, not parity.

**Do not use `benchmark_parity.py --gop 1` for this.** That is how the denominator
was produced, but it rebuilds the whole deployed stack through
`m21.prepare_rate_point` - the M11-G16 context model, the M10K model, the K=512
codebooks - all of which are fitted to a **64-channel** latent. Stage 1's latent has
192, so it raises a provenance error rather than producing a number.
`benchmark_intra_gate.py` exists precisely for this: an I-frame needs only the intra
grid, which `calibrate_grids` fits by running the autoencoder, so nothing has to be
retrained. It is verified to reproduce the denominator byte-for-byte on the deployed
checkpoint (`outputs/benchmarks/intra_gate_baseline/`).

### What is genuinely still open

A **full-video** number does need the G16 context model, the codebooks and the motion
table refitted to the new latent - they are all P-frame machinery and a new transform
invalidates them. That is real work, but it is not needed for this gate, and it is
worth scoping only once Stage 1 clears it.

In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / 'checkpoints' / 'best.pt'))